# AutoClassify on Kaggle or Colab

Runs the pipeline on a free GPU notebook. The expensive part is building the performance database (`perfdb`); everything after it runs fine on CPU.

**Kaggle:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On** (needs a phone-verified account). Use *Save Version → Save & Run All* to run in the background; `/kaggle/working` is saved as the notebook output.

**Colab:** Runtime → Change runtime type → **T4 GPU**. Run the Google Drive cell so results survive disconnects.

## 1. Get the code
Either clone your GitHub copy of the repo, or upload the zip as a Kaggle Dataset and copy it.

In [ ]:
import os, sys, shutil, subprocess
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO_URL = "https://github.com/<your-username>/autoclassify.git"   # <- change this
UPLOADED = "/kaggle/input/autoclassify-code/autoclassify"           # <- or path of an uploaded copy

os.chdir(WORK)
if not os.path.exists("autoclassify"):
    if os.path.exists(UPLOADED):
        shutil.copytree(UPLOADED, "autoclassify")
    else:
        subprocess.run(["git", "clone", REPO_URL, "autoclassify"], check=True)
os.chdir(f"{WORK}/autoclassify")
print(os.getcwd(), os.listdir("."))

## 2. (Colab only) keep results on Google Drive

In [ ]:
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
    store = "/content/drive/MyDrive/autoclassify_results"
    os.makedirs(store, exist_ok=True)
    if not os.path.islink("results"):
        if os.path.exists("results"):
            shutil.copytree("results", store, dirs_exist_ok=True); shutil.rmtree("results")
        os.symlink(store, "results")
    print("results ->", os.path.realpath("results"))

## 3. (Kaggle only) resume from an earlier run
Add the previous notebook version's output as an input, then set its path here. Finished fits are skipped automatically.

In [ ]:
PREVIOUS = "/kaggle/input/<previous-output>/autoclassify"   # <- set after the first run, else leave
if ON_KAGGLE and os.path.exists(PREVIOUS):
    for sub in ("results", "data"):
        if os.path.exists(f"{PREVIOUS}/{sub}"):
            shutil.copytree(f"{PREVIOUS}/{sub}", sub, dirs_exist_ok=True)
    print("resumed from", PREVIOUS)

## 4. Install and check

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU"
!python -c "import sklearn, lightgbm, catboost, pytabkit, tabpfn, gymnasium, pymfe, torch; print('imports ok; cuda =', torch.cuda.is_available())"

In [ ]:
!python -m pytest -q        # ~10 s: unit tests + a full synthetic mini-pipeline

## 5. Choose the stage
`SHARD = '0/1'` runs every dataset. To split Stage 3 over several sessions or accounts, run `'0/3'`, `'1/3'`, `'2/3'` in different notebooks and merge the `results/<stage>/perfdb/*.csv` files (one file per dataset, so merging is just copying).

In [ ]:
CONFIG = "configs/stage1.yaml"
SHARD = "0/1"

## 6. Datasets and splits (run once per stage; the split is frozen afterwards)

In [ ]:
!python -m autoclassify.cli fetch --config $CONFIG
!python -m autoclassify.cli split --config $CONFIG

## 7. Performance database (the long step; resumable)

In [ ]:
!python -m autoclassify.cli perfdb --config $CONFIG --shard $SHARD
!python -m autoclassify.cli status --config $CONFIG

## 8. Dataset features, RL training, evaluation on meta-val, figures (CPU is enough)

In [ ]:
!python -m autoclassify.cli features --config $CONFIG
!python -m autoclassify.cli train --config $CONFIG
!python -m autoclassify.cli evaluate --config $CONFIG
!python -m autoclassify.cli figures --config $CONFIG

## 9. Download everything

In [ ]:
shutil.make_archive(f"{WORK}/autoclassify_outputs", "zip", ".", ".")
print("saved", f"{WORK}/autoclassify_outputs.zip")